### Embedding Models with specific purpose (Strctured Text)
> The way of representing data as structured information (Key value, JSON etc) has implications in the way how embedding models handle it  
> Some models handle them better than the others  

#### Import of required Libraries

In [ ]:
# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Google AI library
from google import genai
from google.genai import types

# Load Environment variables from file
from dotenv import load_dotenv

# Initialise an client object with API key
load_dotenv ()
client = genai.Client()

# Functions from SciPy for check / test
from scipy import spatial

import time

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

#### Utility Functions
> Functions defined for basic comparison and test  
> It can be re-used in different modules

In [ ]:
def cosine_similarity(vec1, vec2):

    """
    Function to provide cosine similarity between given 2 vectors.
    Vectors are provided in the form of list of numbers
    Cosine Similarity is returned as a number. 1 being the highest representing high similarity
    """

    # spatial.distance.cosine returns the cosine distance
    # from the distance similarity can be computed, considering the unit vector
    
    return 1 - spatial.distance.cosine(vec1, vec2)

In [ ]:
def get_cosine_similarities_HF(model, pairs):
    
    """
    Function to compute similarity scores in each pair for a given list of text pairs.
    Model from HF is given as an argument : as a sentence transformer model.
    For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers
    """

    similarities = []
    
    for s1, s2 in pairs:
                
        # Embed the texts
        emb1 = model.encode(s1, convert_to_tensor=True)
        emb2 = model.encode(s2, convert_to_tensor=True)
        
        # Identify the cosine similarity
        sim = util.cos_sim(emb1, emb2).item()
        similarities.append(sim)
    
    return similarities

In [ ]:
def get_cosine_similarities_gemini(model, pairs):
    
    """
    Function to compute similarity scores in each pair for a given list of text pairs.
    Model from Gemini is given as an argument : Name of the embedding model as string.
    For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers    
    """

    similarities = []
    
    for s1, s2 in pairs:
        
        result = client.models.embed_content(
                model=model,        
                contents=[s1, s2],
                config=types.EmbedContentConfig(task_type="CODE_RETRIEVAL_QUERY",output_dimensionality=768)
                )

        sim = cosine_similarity (result.embeddings[0].values, result.embeddings[1].values)

        similarities.append(sim)

        time.sleep (0.5)

    return similarities

#### Make a Similarity Check  
> Identify Structured data and corresponding description in natural language  
> Compute vector for the pair and make similarity check  
> Also, pairs of data and incorrect description is made similarity check  
> This Check is done for Different embedding models from HF

In [ ]:
# Choose 2 embedders from HF to compare
text = "sentence-transformers/all-MiniLM-L6-v2"
str = "BAAI/bge-m3"
model_text = SentenceTransformer(text)
model_str = SentenceTransformer(str)

> Read Code & Text data from CSV file  
> Make them into correct and incorrect pairs of Code Vs Description

In [ ]:
# Load the Test data from CSV file
Similarity_Sample = pd.read_csv ('Str_Data_Embed.csv')

# Text pairs : Code Vs Correct description
Correct_Pairs = list (zip (Similarity_Sample['Structured Info'], Similarity_Sample['Correct Description']))

# Text pairs : Code Vs Incorrect description
Incorrect_Pairs = list (zip (Similarity_Sample['Structured Info'], Similarity_Sample['Incorrect Description']))


#### Text Embedding Model 1

In [ ]:
# Use a Text embedding model
# Calculate pair wise similarity For correct and incorrect descriptions
sims_text_corr = get_cosine_similarities_HF(model_text, Correct_Pairs)
sims_text_incorr = get_cosine_similarities_HF(model_text, Incorrect_Pairs)

# Add to the Dataframe
Similarity_Sample['Sim_Text_Corr'] = sims_text_corr
Similarity_Sample['Sim_Text_Incorr'] = sims_text_incorr

Similarity_Sample.to_csv ('Similarities.csv', index=False, float_format="%.3f")

#### Text Embedding Model 2

In [ ]:
# Use a diff embedding model
# Calculate pair wise similarity For correct and incorrect descriptions
sims_str_corr = get_cosine_similarities_HF(model_str, Correct_Pairs)
sims_str_incorr = get_cosine_similarities_HF(model_str, Incorrect_Pairs)

# Add to the Dataframe
Similarity_Sample['Sim_HF_Str_Corr'] = sims_str_corr
Similarity_Sample['Sim_HF_Str_Incorr'] = sims_str_incorr

Similarity_Sample.to_csv ('Similarities.csv', index=False, float_format="%.3f")